# xgb_3class_xfn5d_simple — Training & Hyperparameter Tuning (XGBoost)

**Purpose**: Same 15-feature explainable set as `lgbm_3class_xfn5d_simple`, but with **XGBoost**:
1. Load dataset, engineer simple features, define constants (identical feature list as LightGBM simple model)
2. Walk-forward hyperparameter search on XGBoost depth / min_child_weight / n_estimators
3. Train production XGBoost on all data, save artifact
4. Re-run 7-fold walk-forward with best XGBoost params **and** with best LightGBM params (loaded from simple LGBM artifact) for a side-by-side performance summary

**Model**: `xgb_3class_xfn5d_simple`  
**Features**: 15 (same as LightGBM simple model)  
**Target**: `target_excess_xfn_5d`  
**Threshold**: ±0.30%


In [1]:
import sys, json, pickle, warnings
from pathlib import Path
from datetime import date
from itertools import product as iterproduct

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path('').resolve()
ROOT         = NOTEBOOK_DIR.parents[1]
EXP_PARENT   = ROOT / 'step3_predictive_model/model_experiments_archive'
ARTIFACT_DIR = NOTEBOOK_DIR / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

for p in [str(ROOT), str(EXP_PARENT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'ROOT         : {ROOT}')
print(f'Artifact dir : {ARTIFACT_DIR}')


ROOT         : /Users/yanyan/Desktop/Projects/AI_driven_company_and_stock_analysis
Artifact dir : /Users/yanyan/Desktop/Projects/AI_driven_company_and_stock_analysis/step3_predictive_model/simple_model/artifacts


## 1. Dataset, Features & Constants

In [2]:
from redesign_single_stock.src.run_redesign_experiments import load_dataset, label_3class
from src.models.walk_forward_config import E10_FOLDS, STRIDE_EVAL

df = load_dataset()
print(f'Dataset : {len(df)} rows  |  {df["date"].min().date()} → {df["date"].max().date()}')

def call_decay_from_days(days: pd.Series, halflife: float = 20.0) -> pd.Series:
    return np.exp(-days.clip(lower=0).fillna(999.0) / halflife)

call_decay = call_decay_from_days(df['days_since_call'])
df['evt_exec_tone'] = (
    df[['transcript_ceo_prep_sentiment_mean_ffill',
        'transcript_cfo_prep_sentiment_mean_ffill']].mean(axis=1) * call_decay
)
df['evt_guidance_topic']     = df['topic_guidance_share_ffill'] * call_decay
df['evt_guidance_sentiment'] = df['topic_guidance_sentiment_ffill'] * call_decay
df['evt_aml_topic']     = df['topic_regulatory_AML_share_ffill'] * call_decay
df['evt_aml_sentiment'] = df['topic_regulatory_AML_sentiment_ffill'] * call_decay

PRICE_FEATURES = ['td_return_5d', 'td_return_20d', 'td_vs_xfn_5d', 'td_dist_52w_high']
NEWS_FEATURES  = ['news_sent_mean_30d', 'news_count_30d', 'days_since_last_news']
TIMING_FEATURES = ['days_since_call', 'is_earnings_week']
NLP_FEATURES = [
    'evt_exec_tone', 'evt_framing_gap',
    'evt_guidance_topic', 'evt_guidance_sentiment',
    'evt_aml_topic', 'evt_aml_sentiment',
]
FEATURES  = PRICE_FEATURES + NEWS_FEATURES + TIMING_FEATURES + NLP_FEATURES
TARGET    = 'target_excess_xfn_5d'
THRESHOLD = 0.003
label_map = {-1: 0, 0: 1, 1: 2}
inv_map   = {0: -1, 1: 0, 2: 1}

STRIDE  = STRIDE_EVAL
OFFSETS = list(range(STRIDE))

missing = [f for f in FEATURES if f not in df.columns]
assert not missing, f'Missing features: {missing}'
print(f'Features : {len(FEATURES)}')


Dataset : 1285 rows  |  2021-02-25 → 2026-04-09
Features : 15


## 2. Helper Functions (same metric definitions as LightGBM notebooks)

In [3]:
def offset_metrics(preds, y_true_cls, y_true_cont, dates):
    rows = []
    for off in OFFSETS:
        idx    = np.arange(off, len(dates), STRIDE)
        p      = preds[idx]
        y_cls  = y_true_cls[idx]
        y_cont = y_true_cont[idx]
        active = (p != 0)
        rows.append({
            'mean_acc':   (p == y_cls).mean(),
            'macro_f1':   f1_score(y_cls, p, average='macro', zero_division=0, labels=[-1, 0, 1]),
            'active_cov': active.mean(),
            'active_asa': float((np.sign(p[active]) == np.sign(y_cont[active])).mean())
                          if active.sum() > 0 else np.nan,
        })
    return pd.DataFrame(rows).mean()


def fit_xgb(train_df, test_df, params: dict):
    x_tr  = train_df[FEATURES].fillna(train_df[FEATURES].median())
    y_tr  = label_3class(train_df[TARGET].values, threshold=THRESHOLD)
    y_enc = np.array([label_map[v] for v in y_tr], dtype=int)
    clf = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        random_state=42,
        n_jobs=1,
        verbosity=0,
        **params,
    )
    clf.fit(x_tr, y_enc)
    x_te      = test_df[FEATURES].fillna(train_df[FEATURES].median())
    y_te_cls  = label_3class(test_df[TARGET].values, threshold=THRESHOLD)
    y_te_cont = test_df[TARGET].values
    preds_enc = clf.predict(x_te)
    preds     = np.array([inv_map[int(p)] for p in preds_enc])
    return clf, preds, y_te_cls, y_te_cont

print('Helpers defined.')


Helpers defined.


## 3. Hyperparameter Search (XGBoost)

Sweep depth, `min_child_weight`, and `n_estimators`. Fixed: `learning_rate=0.05`, `subsample=0.8`, `colsample_bytree=0.8`, `reg_alpha=0.2`, `reg_lambda=1.0`, `tree_method='hist'`.

**Selection criterion**: composite = `0.6 × active_asa + 0.4 × macro_f1` (same as LightGBM simple training).


In [4]:
BASE_XGB = dict(
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.2,
    reg_lambda=1.0,
    tree_method='hist',
)

GRID = {
    'max_depth':         [3, 4, 6],
    'min_child_weight':  [1, 5, 10],
    'n_estimators':      [80, 120, 200],
}

combos = list(iterproduct(GRID['max_depth'], GRID['min_child_weight'], GRID['n_estimators']))
print(f'Running {len(combos)} configs × {len(E10_FOLDS)} folds …\n')

search_rows = []
for i, (depth, mcw, ne) in enumerate(combos, 1):
    params = {**BASE_XGB, 'max_depth': depth, 'min_child_weight': mcw, 'n_estimators': ne}
    fold_ms = []
    for fold_id, train_end, test_start, test_end in E10_FOLDS:
        tr_mask = (df['date'] <= pd.Timestamp(train_end)) & df[TARGET].notna()
        te_mask = ((df['date'] >= pd.Timestamp(test_start)) &
                   (df['date'] <= pd.Timestamp(test_end)) & df[TARGET].notna())
        _, preds, y_cls, y_cont = fit_xgb(df[tr_mask], df[te_mask], params)
        fold_ms.append(offset_metrics(preds, y_cls, y_cont, df[te_mask]['date'].values))
    agg  = pd.DataFrame(fold_ms).mean()
    comp = 0.6 * float(agg.active_asa) + 0.4 * float(agg.macro_f1)
    search_rows.append({
        'max_depth': depth, 'min_child_weight': mcw, 'n_estimators': ne,
        'mean_acc':   round(float(agg.mean_acc),   3),
        'macro_f1':   round(float(agg.macro_f1),   3),
        'active_asa': round(float(agg.active_asa),  3),
        'active_cov': round(float(agg.active_cov),  3),
        'composite':  round(comp, 4),
    })
    print(f'[{i:2d}/{len(combos)}]  depth={depth}  mcw={mcw:2d}  ne={ne:3d}  '
          f'asa={agg.active_asa:.3f}  f1={agg.macro_f1:.3f}  composite={comp:.4f}')

print('\nSearch complete.')


Running 27 configs × 7 folds …



[ 1/27]  depth=3  mcw= 1  ne= 80  asa=0.533  f1=0.310  composite=0.4438


[ 2/27]  depth=3  mcw= 1  ne=120  asa=0.508  f1=0.300  composite=0.4248


[ 3/27]  depth=3  mcw= 1  ne=200  asa=0.506  f1=0.293  composite=0.4206


[ 4/27]  depth=3  mcw= 5  ne= 80  asa=0.522  f1=0.297  composite=0.4324


...

[23/27]  depth=6  mcw= 5  ne=120  asa=0.496  f1=0.289  composite=0.4134


[24/27]  depth=6  mcw= 5  ne=200  asa=0.510  f1=0.302  composite=0.4264


[25/27]  depth=6  mcw=10  ne= 80  asa=0.514  f1=0.312  composite=0.4330


[26/27]  depth=6  mcw=10  ne=120  asa=0.517  f1=0.312  composite=0.4352


[27/27]  depth=6  mcw=10  ne=200  asa=0.501  f1=0.302  composite=0.4210

Search complete.


## 4. Best Configuration & Final Training

In [5]:
search_df = (
    pd.DataFrame(search_rows)
    .sort_values('composite', ascending=False)
    .reset_index(drop=True)
)
search_df.index += 1
display(search_df.head(12))

best_row = search_df.iloc[0]
BEST_XGB = {
    **BASE_XGB,
    'max_depth':         int(best_row.max_depth),
    'min_child_weight': int(best_row.min_child_weight),
    'n_estimators':      int(best_row.n_estimators),
}
print('\n── Best XGBoost configuration ──')
for k, v in BEST_XGB.items():
    print(f'  {k}: {v}')
print(f'\n  Composite : {best_row.composite:.4f}')

# Train on all data
full_df   = df[df[TARGET].notna()].copy()
train_end = full_df['date'].max()
x_full = full_df[FEATURES].fillna(full_df[FEATURES].median())
y_full = label_3class(full_df[TARGET].values, threshold=THRESHOLD)
y_enc  = np.array([label_map[v] for v in y_full], dtype=int)

final_xgb = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3,
    random_state=42, n_jobs=1, verbosity=0,
    **BEST_XGB,
)
final_xgb.fit(x_full, y_enc)
print(f'\nTrained XGB on {len(full_df)} rows up to {train_end.date()}')


,max_depth,min_child_weight,n_estimators,mean_acc,macro_f1,active_asa,active_cov,composite
1,3,1,80,0.415,0.310,0.533,0.976,0.4438
2,3,10,80,0.410,0.310,0.528,0.965,0.4408
3,6,10,120,0.396,0.312,0.517,0.945,0.4352
4,3,5,120,0.399,0.307,0.519,0.962,0.4341
5,4,10,120,0.400,0.310,0.515,0.953,0.4333
6,6,10,80,0.399,0.312,0.514,0.954,0.4330
7,3,5,80,0.401,0.297,0.522,0.971,0.4324
8,4,10,80,0.399,0.302,0.518,0.968,0.4319
9,3,10,120,0.395,0.303,0.513,0.954,0.4294
10,4,10,200,0.386,0.306,0.511,0.933,0.4290



── Best XGBoost configuration ──
  learning_rate: 0.05
  subsample: 0.8
  colsample_bytree: 0.8
  reg_alpha: 0.2
  reg_lambda: 1.0
  tree_method: hist
  max_depth: 3
  min_child_weight: 1
  n_estimators: 80

  Composite : 0.4438

Trained XGB on 1285 rows up to 2026-04-09


## 5. Save XGBoost Artifact

In [6]:
from datetime import date
TODAY       = date.today().isoformat()
ARTIFACT_ID = f'xgb_3class_xfn5d_simple_tuned-{TODAY}'
PKL_PATH    = ARTIFACT_DIR / f'{ARTIFACT_ID}.pkl'
JSON_PATH   = ARTIFACT_DIR / f'{ARTIFACT_ID}.json'

artifact = {
    'model':              final_xgb,
    'features':           FEATURES,
    'target':             TARGET,
    'threshold':          THRESHOLD,
    'label_map':          label_map,
    'inverse_label_map':  inv_map,
    'best_params':        BEST_XGB,
    'train_rows':         len(full_df),
    'train_end':          str(train_end.date()),
    'artifact_id':        ARTIFACT_ID,
}
with open(PKL_PATH, 'wb') as f:
    pickle.dump(artifact, f)

meta = {
    'artifact_id':     ARTIFACT_ID,
    'model_name':      'xgb_3class_xfn5d_simple',
    'artifact_type':   'tuned',
    'created':         TODAY,
    'source':          'xgb_3class_xfn5d_simple_training.ipynb',
    'best_params':     BEST_XGB,
    'search_winner':   {
        'active_asa':  float(best_row.active_asa),
        'macro_f1':    float(best_row.macro_f1),
        'composite':   float(best_row.composite),
    },
    'features':        FEATURES,
    'n_features':      len(FEATURES),
    'target':          TARGET,
    'threshold':       THRESHOLD,
    'train_rows':      int(len(full_df)),
    'train_end':       str(train_end.date()),
    'artifact_path':   str(PKL_PATH),
}
with open(JSON_PATH, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved: {PKL_PATH.name}')
print(f'Saved: {JSON_PATH.name}')


Saved: xgb_3class_xfn5d_simple_tuned-2026-04-23.pkl
Saved: xgb_3class_xfn5d_simple_tuned-2026-04-23.json


## 6. Walk-Forward Performance — XGBoost vs LightGBM (same folds, same 15 features)

Both models are re-fit per fold on the training window only (no look-ahead). LightGBM uses `best_params` from the most recent `lgbm_3class_xfn5d_simple_tuned-*.pkl` artifact.


In [7]:
import lightgbm as lgb
import pickle
from IPython.display import display

# Load LightGBM simple artifact
lgb_pkls = sorted(ARTIFACT_DIR.glob('lgbm_3class_xfn5d_simple_tuned-*.pkl'))
assert lgb_pkls, 'Run lgbm_3class_xfn5d_simple_training.ipynb first to create the LGBM artifact.'
with open(lgb_pkls[-1], 'rb') as f:
    lgb_art = pickle.load(f)
LGB_FEATURES = lgb_art['features']
LGB_PARAMS   = lgb_art['best_params']
assert LGB_FEATURES == FEATURES, 'Feature list mismatch between LGBM and XGB notebooks'


def fit_lgb(train_df, test_df):
    x_tr  = train_df[FEATURES].fillna(train_df[FEATURES].median())
    y_tr  = label_3class(train_df[TARGET].values, threshold=THRESHOLD)
    y_enc = np.array([label_map[v] for v in y_tr], dtype=int)
    clf = lgb.LGBMClassifier(
        objective='multiclass', num_class=3,
        random_state=42, verbose=-1, n_jobs=1,
        **LGB_PARAMS,
    )
    clf.fit(x_tr, y_enc)
    x_te      = test_df[FEATURES].fillna(train_df[FEATURES].median())
    y_te_cls  = label_3class(test_df[TARGET].values, threshold=THRESHOLD)
    y_te_cont = test_df[TARGET].values
    preds_enc = clf.predict(x_te)
    preds     = np.array([inv_map[p] for p in preds_enc])
    return preds, y_te_cls, y_te_cont


def eval_model(name, fit_fn):
    rows = []
    for fold_id, train_end, test_start, test_end in E10_FOLDS:
        tr = (df['date'] <= pd.Timestamp(train_end)) & df[TARGET].notna()
        te = ((df['date'] >= pd.Timestamp(test_start)) &
              (df['date'] <= pd.Timestamp(test_end)) & df[TARGET].notna())
        train_df, test_df = df[tr], df[te]
        if name.startswith('XGB'):
            _, preds, y_cls, y_cont = fit_xgb(train_df, test_df, BEST_XGB)
        else:
            preds, y_cls, y_cont = fit_lgb(train_df, test_df)
        m = offset_metrics(preds, y_cls, y_cont, test_df['date'].values)
        rows.append({'fold': fold_id, **m.to_dict()})
    return pd.DataFrame(rows).set_index('fold')

xgb_wf = eval_model('XGB', None)
lgb_wf = eval_model('LGB', None)

print('=== Per-fold (XGBoost) ===')
display(xgb_wf.round(3))
print('=== Per-fold (LightGBM) ===')
display(lgb_wf.round(3))

agg = pd.DataFrame({
    'xgb_15f_mean':  xgb_wf.mean(),
    'lgbm_15f_mean': lgb_wf.mean(),
})
agg['delta_xgb_minus_lgb'] = agg['xgb_15f_mean'] - agg['lgbm_15f_mean']

print('\n=== 7-fold mean comparison (same evaluation protocol) ===')
display(agg.round(4))

xc = 0.6 * agg.loc['active_asa', 'xgb_15f_mean'] + 0.4 * agg.loc['macro_f1', 'xgb_15f_mean']
lc = 0.6 * agg.loc['active_asa', 'lgbm_15f_mean'] + 0.4 * agg.loc['macro_f1', 'lgbm_15f_mean']
print(f'\nComposite (0.6×ASA + 0.4×F1):  XGB={xc:.4f}   LGBM={lc:.4f}   (Δ={xc-lc:+.4f})')


=== Per-fold (XGBoost) ===


,mean_acc,macro_f1,active_cov,active_asa
fold,,,,
v1,0.371,0.284,0.926,0.536
v2,0.454,0.336,0.992,0.568
v3,0.455,0.337,0.984,0.572
v4,0.471,0.355,0.959,0.567
v5,0.358,0.245,0.983,0.448
v6,0.414,0.334,0.992,0.534
v7,0.383,0.280,1.000,0.506


=== Per-fold (LightGBM) ===


,mean_acc,macro_f1,active_cov,active_asa
fold,,,,
v1,0.313,0.244,0.843,0.517
v2,0.411,0.319,0.932,0.576
v3,0.555,0.421,0.975,0.679
v4,0.487,0.362,0.975,0.559
v5,0.325,0.263,0.892,0.412
v6,0.405,0.368,0.918,0.550
v7,0.399,0.296,0.985,0.500



=== 7-fold mean comparison (same evaluation protocol) ===


,xgb_15f_mean,lgbm_15f_mean,delta_xgb_minus_lgb
mean_acc,0.4152,0.4136,0.0016
macro_f1,0.3099,0.3246,-0.0147
active_cov,0.9764,0.9314,0.0450
active_asa,0.5330,0.5419,-0.0089



Composite (0.6×ASA + 0.4×F1):  XGB=0.4438   LGBM=0.4550   (Δ=-0.0112)


## 7. Summary — Simple 15f XGBoost vs Simple 15f LightGBM

- **Same inputs**: identical 15 features, same `target_excess_xfn_5d`, ±0.30% band, same 7 walk-forward folds and stride-5 offset averaging.
- **Different learners**: tree growth and regularisation differ; hyperparameters were tuned separately for each library (grids are not directly comparable parameter-for-parameter).
- **How to read the table above**: positive `delta_xgb_minus_lgb` means XGBoost scored higher on that metric on average across folds.

If XGBoost underperforms on ASA but matches on F1, the model may be issuing more neutral predictions (check `active_cov`). If both ASA and Dir-relevant metrics move together, the ranking is stable.
